## 1. Install the required packages

In [ ]:
%pip install kagglehub librosa matplotlib ipython opencv-python mediapipe tqdm numpy ffmpeg "imageio[ffmpeg]" joblib pandas transformers
# PyTorch gpu (CUDA 12.6)
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
# PyTorch cpu
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

In [ ]:
from dataclasses import dataclass
from joblib import Parallel, delayed
from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import Wav2Vec2FeatureExtractor, WavLMModel
from IPython.display import Audio, Video, display
from mediapipe.tasks.python import BaseOptions, vision

import copy
import cv2
import imageio_ffmpeg
import imageio.v3 as iio
import kagglehub
import librosa
import librosa.display
import matplotlib.pyplot as plt
import mediapipe as mp
import numpy as np
import os
import pandas as pd
import random
import subprocess
import sys
import torch
import torch.nn as nn


## 2. Download the CREMA-D dataset from Kaggle

In [ ]:
from pathlib import Path

import kagglehub

# Download latest version
path = kagglehub.dataset_download("orvile/crema-d-emotional-multimodal-dataset")
path = Path(path) / "content" / "CREMA-D"

print("Path to dataset files:", path)

## 3. Extract the landmarks from the videos using MediaPipe and save them as .npy files to the `Landmarks` folder.

In [ ]:
from joblib import Parallel, delayed
from tqdm.auto import tqdm
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks.python import BaseOptions, vision

INPUT_VIDEO_DIR = path / "VideoFlash"
OUTPUT_DATA_DIR = Path("Landmarks")
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)


def process_video_worker(video_path):
    npy_path = OUTPUT_DATA_DIR / video_path.with_suffix(".npy").name
    if npy_path.exists():
        return  # Skip already processed files

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"Warning: Could not open video {video_path.name}")
        return

    # Initialize MediaPipe locally inside each worker process
    base_options = BaseOptions(model_asset_path="face_landmarker.task")
    options = vision.FaceLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO,
        output_face_blendshapes=False,
        output_facial_transformation_matrixes=False,
        num_faces=1,
    )
    detector = vision.FaceLandmarker.create_from_options(options)

    video_sequence = []
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0 or np.isnan(fps):
        fps = 30.0

    frame_idx = 0
    timestamp_offset = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or frame is None:
            break

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        current_ts = timestamp_offset + int((frame_idx / fps) * 1000)

        results = detector.detect_for_video(mp_image, current_ts)

        if results.face_landmarks:
            frame_landmarks = [[lm.x, lm.y, lm.z] for lm in results.face_landmarks[0]]
            video_sequence.append(frame_landmarks)
        else:
            video_sequence.append(np.zeros((478, 3)))

        frame_idx += 1

    cap.release()

    if video_sequence:
        np_sequence = np.array(video_sequence, dtype=np.float32)
        np.save(npy_path, np_sequence)


# Run across all CPU cores using joblib
video_files = list(INPUT_VIDEO_DIR.glob("*.flv"))
print(
    f"Extracting landmarks from {len(video_files)} videos using joblib multiprocessing..."
)

# n_jobs=-1 utilizes all available CPU cores safely on Windows
Parallel(n_jobs=-1)(delayed(process_video_worker)(video) for video in tqdm(video_files))

print("Landmark extraction complete!")

## 4. Visualize the results on a random sample

In [ ]:
import os
import random
from pathlib import Path
import cv2
import numpy as np
import imageio.v3 as iio
import subprocess
import imageio_ffmpeg

import librosa
import librosa.display
import matplotlib.pyplot as plt
from IPython.display import Audio, Video, display

AUDIO_PATH = path / "AudioWAV"
VIDEO_PATH = path / "VideoFlash"
LANDMARK_PATH = Path("Landmarks")

data = []
for file in list(LANDMARK_PATH.glob("*.npy")):
    parts = file.stem.split("_")
    data.append(
        {
            "ActorID": parts[0],
            "Sentence": parts[1],
            "Emotion": parts[2],
            "Intensity": parts[3],
            "AudioPath": AUDIO_PATH / f"{file.stem}.wav",
            "VideoPath": VIDEO_PATH / f"{file.stem}.flv",
            "LandmarkPath": file,
        }
    )

sample = random.choice(data)
print(
    f"Loading Multi-Modal Data for Actor: {sample['ActorID']} | Emotion: {sample['Emotion']}"
)
input_flv = str(sample["VideoPath"])
wav_path = str(sample["AudioPath"])

# Load audio to get exact duration
y_audio, sr = librosa.load(wav_path, sr=None)
audio_duration = len(y_audio) / sr

# Load video with OpenCV
cap = cv2.VideoCapture(input_flv)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# Fallback for invalid FPS reading
if fps <= 0 or np.isnan(fps):
    fps = 30.0

print(f"Video loaded. Resolution: {width}x{height}, FPS: {fps:.2f}")

landmarks_seq = np.load(sample["LandmarkPath"])

frames = []
for landmarks in landmarks_seq:
    ret, frame = cap.read()
    if not ret or frame is None:
        break

    # Draw landmarks
    for x, y, _ in landmarks:
        cv2.circle(frame, (int(x * width), int(y * height)), 1, (0, 255, 0), -1)

    # OpenCV uses BGR; convert to RGB for imageio
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frames.append(frame_rgb)

cap.release()

if len(frames) == 0:
    raise RuntimeError(f"Could not read frames from video: {input_flv}")

fps_out = len(frames) / audio_duration

silent_mp4 = "silent.mp4"
final_mp4 = "out_with_audio.mp4"

# Write directly to H.264 MP4 using imageio's internal bindings
iio.imwrite(silent_mp4, frames, fps=fps_out, codec="libx264", pixelformat="yuv420p")

ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()  # works even if ffmpeg isn't on PATH
subprocess.run(
    [
        ffmpeg,
        "-y",
        "-i",
        silent_mp4,
        "-i",
        wav_path,
        "-c:v",
        "copy",
        "-c:a",
        "aac",
        "-shortest",
        final_mp4,
    ],
    check=True,
    capture_output=True,
)

print(f"Successfully saved {len(frames)} frames to {final_mp4}")

display(Video(final_mp4, embed=True))

y, sr = librosa.load(sample["AudioPath"], sr=None)
display(Audio(data=y, rate=sr))
plt.figure(figsize=(10, 3))
librosa.display.waveshow(y, sr=sr, alpha=0.6)
plt.title(f"Audio Waveform - Emotion: {sample['Emotion']}")
plt.tight_layout()
plt.show()

## 5. Display data about the landmarks

In [ ]:
import pandas as pd

# Directory where .npy files are stored
data_dir = Path("./Landmarks")

# Initialize information lists
file_paths = []
emotion_labels = []
actor_ids = []

# Loop over all .npy files
for file_path in data_dir.glob("*.npy"):
    # Get the filename without the .npy extension
    filename_stem = file_path.stem

    # Split the filename by "_"
    parts = filename_stem.split("_")

    # Extract information
    actor_id = parts[0]
    emotion = parts[2]

    # Append the data to the lists
    file_paths.append(str(file_path))
    emotion_labels.append(emotion)
    actor_ids.append(actor_id)

# Create a DataFrame from the lists
df = pd.DataFrame(
    {"file_path": file_paths, "emotion": emotion_labels, "actor_id": actor_ids}
)

# Print out the information
print(f"The number of files: {len(file_paths)}")
print(f"The number of unique actors: {df['actor_id'].nunique()}")
print("\nCounts per emotion:")
print(df["emotion"].value_counts())

df["n_frames"] = df["file_path"].apply(lambda p: np.load(p, mmap_mode="r").shape[0])

print(f"\nThe least amount of frames in one clip: {df['n_frames'].min()}")
print(f"Median of frames in the clips: {df['n_frames'].median()}")
print(f"The most amount of frames in one clip: {df['n_frames'].max()}")

## 6. Detect outliers and exlcude them from the data

In [ ]:
# How many clips are very short or very long?
print("Very short clips - very long clips:")
print((df["n_frames"] < 10).sum(), "-", (df["n_frames"] > 300).sum())

# 1. Find the extremely short file
short_files = df[df["n_frames"] < 10]
print("\n--- Extremely Short File ---")
print(short_files[["file_path", "n_frames", "emotion", "actor_id"]])

# 2. Find the extremely long file
long_files = df[df["n_frames"] > 300]
print("\n--- Extremely Long File ---")
print(long_files[["file_path", "n_frames", "emotion", "actor_id"]])

In [ ]:
# Exclude the outliers
MIN_FRAMES = 10
MAX_FRAMES = 300

outliers = df[(df["n_frames"] < MIN_FRAMES) | (df["n_frames"] > MAX_FRAMES)]
print("Dropping: ", len(outliers), "clips")

# Create a dataframe without outliers
df_clean = df[
    (df["n_frames"] >= MIN_FRAMES) & (df["n_frames"] <= MAX_FRAMES)
].reset_index(drop=True)
print(len(df), "->", len(df_clean))

## 7. Normalization and resampling for each clip

In [ ]:
def normalize_clip(x):
    """Remove head position and face size from every frame of a landmark clip.

    Each frame is handled independently, so only the shape of the face remains.

    Parameters
    ----------
    x : np.ndarray, shape (N, 478, 3)
        N frames, 478 face landmarks, (x, y, z) coordinates per landmark.

    Returns
    -------
    np.ndarray, shape (N, 478, 3)
        Centered and scaled landmarks.
    """
    # Remove position: subtract each frame's mean landmark, so the face center is at the origin
    # Results in a shape of (N, 1, 3)
    center = np.mean(x, axis=1, keepdims=True)
    x_centered = x - center

    # Remove scale: RMS distance of the landmakrs from the center, one number per frame.
    # Results in a shape of (N, 1, 1)
    scale = np.sqrt(np.mean(x_centered**2, axis=(1, 2), keepdims=True))
    return x_centered / (scale + 1e-8)  # Guards against division by zero

In [ ]:
def resample_clip(x, num_frames):
    """Resample a (N, 478, 3) clip to num_frames evenly spaced frames.

    If num_frames is None or <= 0, the clip is returned unchanged.
    Returns (clip, idx), where idx are the frame indices used.
    """
    if not num_frames or num_frames <= 0:
        return x, np.arange(len(x))
    idx = np.linspace(0, len(x) - 1, num_frames).round().astype(int)
    return x[idx], idx

In [ ]:
NUM_FRAMES = 64  # Resampling size of each clip (SET TO 0 IF NO RESAMPLING IS WANTED)
emotion_map = {"ANG": 0, "DIS": 1, "FEA": 2, "HAP": 3, "SAD": 4, "NEU": 5}

# Load, normalize and resample each clip
clips = []
for path in df_clean["file_path"]:
    x = np.load(path)  # (N, 478, 3), N(number of frames) varies between clips
    x = normalize_clip(x)  # Remove position and scale per frame

    # Resampling
    x, _ = resample_clip(x, NUM_FRAMES)  # (NUM_FRAMES, 478, 3)
    clips.append(x)

# Build dataset arrays
if NUM_FRAMES:
    X = np.stack(clips)  # (n_clips, NUM_FRAMES, 487, 3)
else:
    X = clips  # list of (N_i, 478, 3) arrays
y = df_clean["emotion"].map(emotion_map).to_numpy()  # emotion as integer 0-5
groups = df_clean["actor_id"].to_numpy()  # actor IDs

# Sanity checks
print("Dataset structure:")
if NUM_FRAMES:
    print("X shape:", X.shape)
else:
    print(f"X is a list of {len(X)} clips with variable lengths.")
    print(f"First clip shape: {X[0].shape}, Last clip shape: {X[-1].shape}")

print("Emotions: ", y.shape, "Actors: ", groups.shape)
print("\nClips per emotion class:")
print(np.bincount(y))

In [ ]:
# Save normalized and resampled clips

from pathlib import Path

OUT_DIR = Path(f"Landmarks_normalized_{NUM_FRAMES}")
OUT_DIR.mkdir(exist_ok=True)

for path in df_clean["file_path"]:
    name = Path(path).name
    if (OUT_DIR / name).exists():
        continue
    x = np.load(path)
    x = normalize_clip(x).astype(np.float32)

    # Save normalized + resampled
    x_rs, _ = resample_clip(x, NUM_FRAMES)
    np.save(OUT_DIR / name, x_rs)

print(f"Saved {len(list(OUT_DIR.glob('*.npy')))} files")


## 7. Display the result of normalization and resampling

In [ ]:
def render_landmarks(clip, size=400, extent=3.0):
    """Draw each frame of a (N, 478, 3) normalized clip as dots on a black canvas.

    extent: half-width of the visible area in normalized units.
    The face has RMS size 1, so extent=3 leaves some margin.
    """
    frames = []
    for lm in clip:
        canvas = np.zeros((size, size, 3), dtype=np.uint8)
        # map [-extent, extent] -> [0, size]; flip y because image y points down
        px = ((lm[:, 0] / extent + 1) / 2 * size).astype(int)
        py = ((lm[:, 1] / extent + 1) / 2 * size).astype(int)
        for u, v in zip(px, py):
            if 0 <= u < size and 0 <= v < size:
                cv2.circle(canvas, (u, v), 1, (0, 255, 0), -1)
        frames.append(canvas)
    return frames


# Pick a random clip from the processed dataset
i = random.randrange(len(X))
print(
    "Emotion:",
    df_clean["emotion"].iloc[i],
    "| Actor:",
    df_clean["actor_id"].iloc[i],
    "| File:",
    df_clean["file_path"].iloc[i],
)

norm_frames = render_landmarks(X[i])

# Raw clip for comparison: same frames, same canvas scheme, but only centered and scaled by a constant
raw = np.load(df_clean["file_path"].iloc[i])
if NUM_FRAMES > 0:
    raw = raw[np.linspace(0, len(raw) - 1, NUM_FRAMES).round().astype(int)]
raw_centered = (
    raw - raw.mean(axis=(0, 1), keepdims=True)
) / raw.std()  # one fixed shift/scale for the whole clip
raw_frames = render_landmarks(raw_centered)

# Put raw and normalized side by side
side_by_side = [np.hstack((r, n)) for r, n in zip(raw_frames, norm_frames)]

iio.imwrite(
    "normalized_check.mp4", side_by_side, fps=15, codec="libx264", pixelformat="yuv420p"
)
display(Video("normalized_check.mp4", embed=True))

In [ ]:
import subprocess
import imageio_ffmpeg


def make_comparison(i, out_file="compare_audio.mp4"):
    """Original video + raw landmarks (left), normalized/resampled landmarks (right), with audio."""
    path = df_clean["file_path"].iloc[i]
    stem = Path(path).stem
    print("Clip:", stem, "| Emotion:", df_clean["emotion"].iloc[i])

    # --- Frame indices used for resampling (same as in the dataset) ---
    raw = np.load(path)  # (N, 478, 3)
    norm, idx = resample_clip(normalize_clip(raw), NUM_FRAMES)

    # --- Left panel: original video frames with raw landmarks ---
    cap = cv2.VideoCapture(str(VIDEO_PATH / f"{stem}.flv"))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    video_frames = []
    while True:
        ret, f = cap.read()
        if not ret:
            break
        video_frames.append(f)
    cap.release()

    left = []
    for k in idx:
        f = video_frames[
            min(k, len(video_frames) - 1)
        ].copy()  # guard against length mismatch
        for x, y, _ in raw[k]:
            cv2.circle(f, (int(x * w), int(y * h)), 1, (0, 255, 0), -1)
        f = cv2.resize(f, (400, 400))
        left.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))

    # --- Right panel: saved normalized + resampled clip on a blank canvas ---
    right = render_landmarks(norm)

    side_by_side = [np.hstack((l, r)) for l, r in zip(left, right)]

    # --- Match video duration to the audio duration ---
    wav_path = AUDIO_PATH / f"{stem}.wav"
    y_audio, sr = librosa.load(wav_path, sr=None)
    duration = len(y_audio) / sr
    if NUM_FRAMES > 0:
        fps_out = NUM_FRAMES / duration
    else:
        fps_out = len(norm) / duration

    # --- Write silent video, then add the audio track ---
    iio.imwrite(
        "compare_silent.mp4",
        side_by_side,
        fps=fps_out,
        codec="libx264",
        pixelformat="yuv420p",
    )

    ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()  # works even if ffmpeg isn't on PATH
    subprocess.run(
        [
            ffmpeg,
            "-y",
            "-i",
            "compare_silent.mp4",
            "-i",
            str(wav_path),
            "-c:v",
            "copy",
            "-c:a",
            "aac",
            "-shortest",
            out_file,
        ],
        check=True,
        capture_output=True,
    )

    display(Video(out_file, embed=True))

    # Same as your original code: separate audio player + waveform
    display(Audio(data=y_audio, rate=sr))
    plt.figure(figsize=(10, 3))
    librosa.display.waveshow(y_audio, sr=sr, alpha=0.6)
    plt.title(f"Audio Waveform - Emotion: {df_clean['emotion'].iloc[i]}")
    plt.tight_layout()
    plt.show()


# Random clip:
make_comparison(random.randrange(len(df_clean)))

## WavLM Audio Features

In [ ]:
import librosa
import numpy as np
import torch
from tqdm.auto import tqdm
from transformers import Wav2Vec2FeatureExtractor, WavLMModel

MODEL_ID = "microsoft/wavlm-base"
TARGET_SR = 16_000
FEATURE_DIR = Path("AudioFeatures")
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

records_to_process = [
    (
        record,
        FEATURE_DIR / f"{Path(record['AudioPath']).stem}.npy",
    )
    for record in data
    if not (FEATURE_DIR / f"{Path(record['AudioPath']).stem}.npy").exists()
]

if records_to_process:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_ID)
    model = WavLMModel.from_pretrained(MODEL_ID).to(device)
    model.eval()

    for record, output_path in tqdm(
        records_to_process, desc="Extracting WavLM embeddings", unit="audio"
    ):
        waveform, _ = librosa.load(
            record["AudioPath"],
            sr=TARGET_SR,
            mono=True,
        )
        if waveform.size == 0:
            raise ValueError(f"Empty audio file: {record['AudioPath']}")

        inputs = feature_extractor(
            waveform,
            sampling_rate=TARGET_SR,
            return_tensors="pt",
            return_attention_mask=True,
        )
        inputs = {name: tensor.to(device) for name, tensor in inputs.items()}

        with torch.inference_mode():
            frame_features = model(**inputs).last_hidden_state[0]

        mean = frame_features.mean(dim=0)
        std = frame_features.std(dim=0, unbiased=False)
        embedding = torch.cat((mean, std)).cpu().numpy()
        np.save(output_path, np.asarray(embedding, dtype=np.float32))

    print(f"Saved {len(records_to_process)} audio feature files to {FEATURE_DIR.resolve()}")
else:
    print(f"Audio feature files already exist in {FEATURE_DIR}; skipping extraction.")

## Training

In [ ]:
import copy
import os
import random
import sys
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

In [ ]:
@dataclass
class TrainConfig:
    landmark_dir: Path = Path("Landmarks_normalized_64")
    audio_dir: Path = Path("AudioFeatures")

    visual_input_size: int = 478 * 3
    audio_input_size: int = 1536

    lstm_hidden_size: int = 256
    num_lstm_layers: int = 2

    num_classes: int = 6
    dropout_rate: float = 0.0

    epochs: int = 50
    batch_size: int = 16

    learning_rate: float = 1e-3
    weight_decay: float = 0

    label_smoothing: float = 0.0

    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    pin_memory: bool = torch.cuda.is_available()

In [ ]:
class MultimodalLSTMModel(nn.Module):
    def __init__(self, cfg: TrainConfig, bidirectional: bool = False):
        super().__init__()

        self.visual_lstm = nn.LSTM(
            input_size=cfg.visual_input_size,
            hidden_size=cfg.lstm_hidden_size,
            num_layers=cfg.num_lstm_layers,
            batch_first=True,
            dropout=cfg.dropout_rate,
            bidirectional=bidirectional,
        )
        lstm_output_size = cfg.lstm_hidden_size * (2 if bidirectional else 1)
        self.audio_proj = nn.Sequential(
            nn.Linear(cfg.audio_input_size, lstm_output_size),
            nn.BatchNorm1d(lstm_output_size),
            nn.ReLU(),
            nn.Dropout(p=cfg.dropout_rate),
        )
        self.classifier = nn.Sequential(
            nn.Linear(lstm_output_size * 2, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=cfg.dropout_rate),
            nn.Linear(128, cfg.num_classes),
        )

    def _extract_lstm_embedding(self, lstm, x, lengths):
        packed = pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (hidden, _) = lstm(packed)
        if lstm.bidirectional:
            return torch.cat((hidden[-2], hidden[-1]), dim=1)
        return hidden[-1]

    def forward(self, vis, vis_lens, aud):
        v_embed = self._extract_lstm_embedding(self.visual_lstm, vis, vis_lens)
        a_embed = self.audio_proj(aud)
        combined = torch.cat((v_embed, a_embed), dim=1)
        return self.classifier(combined)

In [ ]:
class EmotionDataset(Dataset):
    def __init__(self, config: TrainConfig, file_list: list):
        self.config = config
        self.emotion_map = {"ANG": 0, "DIS": 1, "FEA": 2, "HAP": 3, "NEU": 4, "SAD": 5}

        self.visuals = []
        self.audios = []
        self.labels = []

        # Pre-load all files into system RAM once before training starts
        for npy_path in tqdm(file_list, desc="Caching dataset to RAM", file=sys.stdout):
            visual_data = np.load(npy_path).astype(np.float32)
            self.visuals.append(
                torch.from_numpy(visual_data.reshape(visual_data.shape[0], -1))
            )

            audio_path = self.config.audio_dir / f"{npy_path.stem}.npy"
            audio_data = np.load(audio_path).astype(np.float32)
            self.audios.append(torch.from_numpy(audio_data))

            try:
                emotion_str = npy_path.stem.split("_")[2].upper()
                label_idx = self.emotion_map.get(emotion_str, 4)
            except IndexError:
                label_idx = 4
            self.labels.append(torch.tensor(label_idx, dtype=torch.long))

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.visuals[idx], self.audios[idx], self.labels[idx]

In [ ]:
def collate_fn(batch):
    visuals, audios, labels = zip(*batch)
    vis_lens = torch.tensor([v.size(0) for v in visuals], dtype=torch.long)
    padded_visuals = pad_sequence(visuals, batch_first=True)
    stacked_audios = torch.stack(audios)
    labels = torch.stack(labels)
    return padded_visuals, vis_lens, stacked_audios, labels


def make_data_split(seed=42):
    cfg = TrainConfig()
    all_files = list(cfg.landmark_dir.glob("*.npy"))
    if not all_files:
        raise FileNotFoundError(f"No .npy files found in {cfg.landmark_dir}")

    all_files = [f for f in all_files if (cfg.audio_dir / f"{f.stem}.npy").exists()]
    if not all_files:
        raise FileNotFoundError(
            f"No matching audio .npy files found in {cfg.audio_dir}"
        )

    rng = random.Random(seed)
    rng.shuffle(all_files)
    n_total = len(all_files)
    train_end = int(0.7 * n_total)
    val_end = int(0.9 * n_total)

    train_files = all_files[:train_end]
    val_files = all_files[train_end:val_end]
    test_files = all_files[val_end:]

    print(
        f"Data Split -> Train: {len(train_files)} | Validation: {len(val_files)} | Test: {len(test_files)}"
    )
    return train_files, val_files, test_files


def train(model_class, train_files, val_files, test_files, model_name="Model", bidirectional=False):
    cfg = TrainConfig()
    print(f"\n{'=' * 20} Training {model_name} {'=' * 20}")
    print(f"Initializing training on: {cfg.device}")

    print(
        f"Data Split -> Train: {len(train_files)} | Validation: {len(val_files)} | Test: {len(test_files)}"
    )

    train_dataset = EmotionDataset(cfg, train_files)
    val_dataset = EmotionDataset(cfg, val_files)
    test_dataset = EmotionDataset(cfg, test_files)

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        pin_memory=cfg.pin_memory,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=cfg.pin_memory,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=cfg.pin_memory,
    )

    # Dynamic model initialization
    model = model_class(cfg, bidirectional).to(cfg.device)
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=3
    )

    best_val_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        current_lr = optimizer.param_groups[0]["lr"]

        with tqdm(
            total=len(train_loader), desc=f"Epoch {epoch}/{cfg.epochs}", file=sys.stdout
        ) as pbar:
            for vis, vis_lens, aud, labels in train_loader:
                vis, aud, labels = (
                    vis.to(cfg.device),
                    aud.to(cfg.device),
                    labels.to(cfg.device),
                )

                optimizer.zero_grad()
                outputs = model(vis, vis_lens, aud)
                loss = criterion(outputs, labels)

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
                optimizer.step()

                train_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                train_total += labels.size(0)
                train_correct += (preds == labels).sum().item()

                pbar.set_postfix(
                    {
                        "Loss": f"{loss.item():.4f}",
                        "Acc": f"{(train_correct / train_total) * 100:.1f}%",
                        "LR": f"{current_lr:g}",
                    }
                )
                pbar.update(1)

            model.eval()
            val_correct, val_total = 0, 0
            with torch.no_grad():
                for vis, vis_lens, aud, labels in val_loader:
                    vis, aud, labels = (
                        vis.to(cfg.device),
                        aud.to(cfg.device),
                        labels.to(cfg.device),
                    )
                    outputs = model(vis, vis_lens, aud)

                    _, preds = torch.max(outputs, 1)
                    val_total += labels.size(0)
                    val_correct += (preds == labels).sum().item()

            val_acc = val_correct / val_total if val_total > 0 else 0

            pbar.set_postfix(
                {
                    "Loss": f"{train_loss / len(train_loader):.4f}",
                    "Acc": f"{(train_correct / train_total) * 100:.1f}%",
                    "Val_acc": f"{val_acc * 100:.1f}%",
                    "LR": f"{current_lr:g}",
                }
            )
            pbar.refresh()

        scheduler.step(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())

    # --- TESTING PHASE ---
    print("\n" + "=" * 50)
    print(
        f"[{model_name}] Training Complete. Best Validation Acc:"
        f" {best_val_acc * 100:.2f}%"
    )
    print("Evaluating Best Model on the 10% Test Set...")
    print("=" * 50)

    model.load_state_dict(best_model_wts)
    model.eval()

    test_preds, test_labels = [], []
    with torch.no_grad():
        for vis, vis_lens, aud, labels in test_loader:
            vis, aud = vis.to(cfg.device), aud.to(cfg.device)
            outputs = model(vis, vis_lens, aud)
            _, preds = torch.max(outputs, 1)

            test_preds.extend(preds.cpu().numpy())
            test_labels.extend(labels.numpy())

    class_names = ["ANG (0)", "DIS (1)", "FEA (2)", "HAP (3)", "NEU (4)", "SAD (5)"]
    test_acc = accuracy_score(test_labels, test_preds)

    print("\n--- TEST CLASSIFICATION REPORT ---")
    print(
        classification_report(
            test_labels, test_preds, target_names=class_names, zero_division=0
        )
    )

    print("--- TEST CONFUSION MATRIX ---")
    cm = confusion_matrix(test_labels, test_preds)
    print(f"{'True \\ Pred':>12} | " + " | ".join([f"{n[:3]:>5}" for n in class_names]))
    print("-" * 65)
    for i, row in enumerate(cm):
        print(f"{class_names[i][:3]:>12} | " + " | ".join([f"{val:>5}" for val in row]))
    print("\n")

    acc_str = f"{test_acc * 100:.1f}".replace(".", "")
    save_dir = Path("models")
    save_dir.mkdir(parents=True, exist_ok=True)

    save_path = save_dir / f"{model_name.lower()}_acc_{acc_str}.pth"
    torch.save(best_model_wts, save_path)
    print(f"Test Accuracy: {test_acc * 100:.2f}%")
    print(f"Model successfully saved to: {save_path}")

In [ ]:
train_files, val_files, test_files = make_data_split()


In [ ]:
if __name__ == "__main__":
    train(
        MultimodalLSTMModel,
        train_files,
        val_files,
        test_files,
        model_name="LSTM-Model",
        bidirectional=False
    )

In [ ]:
if __name__ == "__main__":
    train(
        MultimodalLSTMModel,
        train_files,
        val_files,
        test_files,
        model_name="LSTM-Model-Bi",
        bidirectional=True
    )

## Training: GRU (Gated Recurrent Unit) Model

In [ ]:
class MultimodalGRUModel(nn.Module):
    def __init__(self, cfg: TrainConfig, bidirectional: bool = False):
        super().__init__()
        self.visual_gru = nn.GRU(
            input_size=cfg.visual_input_size,
            hidden_size=cfg.lstm_hidden_size,
            num_layers=cfg.num_lstm_layers,
            batch_first=True,
            dropout=cfg.dropout_rate,
            bidirectional=bidirectional,
        )

        # Note: bidirectional doubles the hidden size output (hidden_size * 2)
        gru_output_size = cfg.lstm_hidden_size * (2 if bidirectional else 1)

        self.audio_proj = nn.Sequential(
            nn.Linear(cfg.audio_input_size, gru_output_size),
            nn.BatchNorm1d(gru_output_size),
            nn.ReLU(),
            nn.Dropout(p=cfg.dropout_rate),
        )

        self.classifier = nn.Sequential(
            nn.Linear(
                gru_output_size * 2, 128
            ),  # visual output + audio output combined
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=cfg.dropout_rate),
            nn.Linear(128, cfg.num_classes),
        )

    def _extract_gru_embedding(self, gru, x, lengths):
        packed = pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, hidden = gru(packed)
        # For bidirectional GRU, combine the final forward and backward hidden layers
        if gru.bidirectional:
            # hidden shape: (num_layers * num_directions, batch, hidden_size)
            h_forward = hidden[-2]
            h_backward = hidden[-1]
            return torch.cat((h_forward, h_backward), dim=1)
        else:
            return hidden[-1]

    def forward(self, vis, vis_lens, aud):
        v_embed = self._extract_gru_embedding(self.visual_gru, vis, vis_lens)
        a_embed = self.audio_proj(aud)
        combined = torch.cat((v_embed, a_embed), dim=1)
        return self.classifier(combined)

In [ ]:
train(MultimodalGRUModel, train_files, val_files, test_files, model_name="GRU-Model-Uni")

In [ ]:
train(MultimodalGRUModel, train_files, val_files, test_files, model_name="GRU-Model", bidirectional=True)

#### Classify one sample using the trained model

In [ ]:
# model_cls = MultimodalLSTMModel
# model_path = "models/model_acc_769.pth"
model_cls = MultimodalGRUModel
model_path = "models/gru-model_acc_781.pth"


def predict_sample(model_cls, model_path, sample=None):
    cfg = TrainConfig()
    model = model_cls(cfg).to(cfg.device)
    model.load_state_dict(torch.load(model_path, map_location=cfg.device))
    model.eval()

    if sample is None:
        sample = random.choice(data)

    stem = sample["LandmarkPath"].stem
    classes = ["ANG", "DIS", "FEA", "HAP", "NEU", "SAD"]

    vis_np = np.load(f"Landmarks_normalized_64/{stem}.npy").astype(np.float32)
    aud_np = np.load(f"AudioFeatures/{stem}.npy").astype(np.float32)

    vis = torch.from_numpy(vis_np.reshape(len(vis_np), -1)).unsqueeze(0).to(cfg.device)
    vis_lens = torch.tensor([len(vis_np)], device=cfg.device)
    aud = torch.from_numpy(aud_np).unsqueeze(0).to(cfg.device)

    with torch.no_grad():
        pred_idx = model(vis, vis_lens, aud).argmax(dim=1).item()

    print(f"Actual: {sample['Emotion']} | Predicted: {classes[pred_idx]}")

    out_mp4 = "preview.mp4"
    subprocess.run(
        [
            imageio_ffmpeg.get_ffmpeg_exe(),
            "-y",
            "-i",
            str(sample["VideoPath"]),
            "-i",
            str(sample["AudioPath"]),
            "-c:v",
            "libx264",
            "-pix_fmt",
            "yuv420p",
            "-c:a",
            "aac",
            "-shortest",
            out_mp4,
        ],
        check=True,
        capture_output=True,
    )

    display(Video(out_mp4, embed=True))
    display(Audio(filename=str(sample["AudioPath"])))


predict_sample(model_cls, model_path)


In [ ]:
def benchmark_model(model_cls, checkpoint_path, test_files, model_name="Model", bidirectional=False):
    cfg = TrainConfig()
    model = model_cls(cfg, bidirectional).to(cfg.device)
    model.load_state_dict(torch.load(checkpoint_path, map_location=cfg.device))
    model.eval()

    if not test_files:
        raise ValueError("No test files available for benchmarking.")

    test_dataset = EmotionDataset(cfg, test_files)
    test_loader = DataLoader(
        test_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=cfg.pin_memory,
    )

    test_labels = []
    test_preds = []

    with torch.no_grad():
        for vis, vis_lens, aud, labels in test_loader:
            vis = vis.to(cfg.device)
            aud = aud.to(cfg.device)
            labels = labels.to(cfg.device)
            outputs = model(vis, vis_lens, aud)
            _, preds = torch.max(outputs, 1)
            test_labels.extend(labels.cpu().numpy())
            test_preds.extend(preds.cpu().numpy())

    class_names = ["ANG", "DIS", "FEA", "HAP", "NEU", "SAD"]
    acc = accuracy_score(test_labels, test_preds)

    print("\n" + "=" * 60)
    print(f"[{model_name}] Benchmark on held-out test set")
    print(f"Accuracy: {acc * 100:.2f}%")
    print("=" * 60)
    print(
        classification_report(
            test_labels, test_preds, target_names=class_names, zero_division=0
        )
    )
    print("Confusion Matrix:")
    print(confusion_matrix(test_labels, test_preds))



# Example usage for an already trained model:
benchmark_model(
    MultimodalLSTMModel, 
    "models/lstm-model_acc_747.pth", 
    test_files, 
    model_name="LSTM-Model-Uni",
)
benchmark_model(
    MultimodalLSTMModel, 
    "models/lstm-model-bi_acc_763.pth", 
    test_files, 
    model_name="LSTM-Model-Bi",
    bidirectional=True
)
benchmark_model(
    MultimodalGRUModel,
    "models/gru-model-uni_acc_734.pth",
    test_files,
    model_name="GRU-Model-Uni",
)
benchmark_model(
    MultimodalGRUModel,
    "models/gru-model_acc_765.pth",
    test_files,
    model_name="GRU-Model-Bi",
    bidirectional=True
)
